In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="SeyedAli/Persian-Speech-Dataset", 
    repo_type="dataset", local_dir="./Persian-Speech-Dataset", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:02<00:00,  1.06it/s]


'/home/ubuntu/Persian-Speech-Dataset'

In [3]:
files = glob('Persian-Speech-Dataset/*/*.parquet')
len(files)

3

In [4]:
df = pd.read_parquet(files[0])
df.head()

,audio_id,audio,speaker_id,gender,emotion,transcript,ipa
0,F21S17,{'bytes': b'RIFF4(\x02\x00WAVEfmt \x10\x00\x00...,F21,female,sad,از زندان بردنش.,ʔæz zendɑn bordæneʃ
1,M53A13,{'bytes': b'RIFF\xf8_\x05\x00WAVEfmt \x10\x00\...,M53,male,angry,وصلت با یک بیگانه یعنی انهدام مملکت و حکومت چین,væslæt bɑ yek bigɑne yæʔni ʔenhedɑme mæmlekæt ...
2,M12N84,{'bytes': b'RIFF>\xd9\x02\x00WAVEfmt \x10\x00\...,M12,male,neutral,ممکنه تو اروپا خریده باشنش,momkene tu ʔorupɑ xӕride bɑʃӕneʃ
3,F01S21,{'bytes': b'RIFFn\x81\x07\x00WAVEfmt \x10\x00\...,F01,female,sad,خیلی دوست داشت یه روزی بره کربلا، اونوقت‎ها هم...,xeyli dust dɑʃt ye ruzi bere kӕrbӕlɑ ʔun vӕxtɑ...
4,M12A51,"{'bytes': b'RIFF""\x15\x03\x00WAVEfmt \x10\x00\...",M12,male,angry,متشکرم برادر عزیز,motʃtʃækeræm bærɑdære ʔæziz


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcript'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1135/1135 [00:59<00:00, 19.17it/s]


In [7]:
with open('Persian-Speech-Dataset.json', 'w') as fopen:
    json.dump(data, fopen)

In [8]:
audio_files = [d['audio_filename'] for d in data]

with open('Persian-Speech-Dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Persian-Speech-Dataset_audio/Persian-Speech-Dataset-data-train-00000-of-00002-f4f130b01ef384ff_0.mp3',
 'text': 'از زندان بردنش.',
 'speaker': 'Persian-Speech-Dataset_audio_F21'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Persian-Speech-Dataset')


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 592.08ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  188kB /  188kB,   ???B/s  
Processing Files (1 / 1): 100%|██████████|  188kB /  188kB,  0.00B/s  
New Data Upload: 100%|██████████|  188kB /  188kB,  0.00B/s  

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.72 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/98d18c38c84f9fa45a2d4ef25cdcb5a61e7ce557', commit_message='Upload dataset', commit_description='', oid='98d18c38c84f9fa45a2d4ef25cdcb5a61e7ce557', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [15]:
# !zip -rq Persian-Speech-Dataset_audio.zip Persian-Speech-Dataset_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS Persian-Speech-Dataset_audio.zip --repo-type=dataset